**Install** **and** **Set** **Up** **PySpark**

> We are dealing with a large Dataset. If we use standard Python (Pandas), the Colab RAM will crash and freeze. PySpark allows us to process this data efficiently by splitting it into chunks in memory.



In [ ]:
# Install Java and PySpark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Download Spark from the official archive (more reliable than dlcdn mirrors)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extract the downloaded file
!tar xf spark-3.5.1-bin-hadoop3.tgz

# Install findspark
!pip install -q findspark

import os
import findspark

# Set environment variables for Spark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
findspark.init()

from pyspark.sql import SparkSession

# Initialize the Spark Session
spark = SparkSession.builder.master("local[*]").appName("RetailMart_Data_Engineer").getOrCreate()
print("Spark Session Successfully Created!")

Spark Session Successfully Created!


**Load** **the** **Raw** **Data**

We need to load the CSV into a Spark DataFrame. We use inferSchema=True so Spark automatically figures out which columns are numbers (integers/doubles) and which are text (strings). We also print the row count and schema to prove the data is loaded correctly.

In [ ]:

df_raw = spark.read.csv("retailmart_transactions.csv", header=True, inferSchema=True)

# Count the rows (This forces Spark to actually read the file)
print(f" Total Rows Loaded: {df_raw.count():,}")

# Show the first 5 rows so we can see the messy data
df_raw.show(5)

# Print the Schema (Data Types)
df_raw.printSchema()

 Total Rows Loaded: 1,000,000
+--------------+--------------------+-------------------+----------------+-------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+
|transaction_id|         customer_id|   transaction_date|product_category| product_name|quantity|unit_price|total_amount|store_location|payment_method|customer_age|customer_gender|loyalty_years|num_visits|churn|
+--------------+--------------------+-------------------+----------------+-------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+
|  TXN-00000001|2a43c044-d9c1-414...|2022-04-03 10:47:06|        Clothing|      T-Shirt|       1|     43.13|       43.13|    Dallas, TX|    Debit Card|        49.0|     Non-Binary|            3|        12|   No|
|  TXN-00000002|717f6db9-0043-480...|2021-03-03 09:13:14|        Clothing|        Dress|       1|     55.36|       55.36| 

**Data** **Quality** **Assessment**

> We must quantify the damage. We need to know exactly how many nulls, duplicates, and invalid dates exist so you can document them in your Executive Report and prove your cleaning strategy worked.



In [ ]:

from pyspark.sql.functions import col, count, when, to_timestamp

print(" DATA QUALITY ASSESSMENT \n")

# Check for Nulls in all columns
print(" Null Counts per Column:")
null_counts = df_raw.select([count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns])
null_counts.show()

#Check for Duplicate transaction_ids
print("Duplicate transaction_ids:")
total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates(["transaction_id"]).count()
print(f"Total Rows: {total_rows:,}")
print(f"Unique Transactions: {unique_rows:,}")
print(f"Duplicates to remove: {total_rows - unique_rows:,}\n")

# Check for Inconsistent Churn Values
# The generator put 'Y', 'N', 'TRUE', 'FALSE', '1', '0'. The Data Scientist needs clean 'Yes'/'No'.
print("Inconsistent 'churn' values:")
df_raw.groupBy("churn").count().show()

# Check for Invalid Dates
print(" Invalid transaction_dates:")
invalid_dates = df_raw.filter(to_timestamp(col("transaction_date"), "yyyy-MM-dd HH:mm:ss").isNull()).count()
print(f"Rows with invalid date formats: {invalid_dates:,}\n")

#Check for Impossible Values (Negative ages, out-of-bounds)
print(" Impossible Values:")
neg_age = df_raw.filter(col("customer_age") < 0).count()
neg_amount = df_raw.filter(col("total_amount") < 0).count()
old_age = df_raw.filter(col("customer_age") > 120).count()
print(f"Negative Ages: {neg_age:,}")
print(f"Negative Amounts: {neg_amount:,}")
print(f"Ages > 120 (Data entry errors): {old_age:,}")

 DATA QUALITY ASSESSMENT 

 Null Counts per Column:
+--------------+-----------+----------------+----------------+------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+
|transaction_id|customer_id|transaction_date|product_category|product_name|quantity|unit_price|total_amount|store_location|payment_method|customer_age|customer_gender|loyalty_years|num_visits|churn|
+--------------+-----------+----------------+----------------+------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+
|             0|       5087|               0|               0|           0|       0|         0|        9845|          4950|         20103|       29756|          48956|            0|         0|    0|
+--------------+-----------+----------------+----------------+------------+--------+----------+------------+--------------+--------------+------------+-

**The** **Cleaning** **Pipeline**

In [ ]:
#  Data Cleaning
from pyspark.sql.functions import lower, trim, median

print("Starting Data Cleaning...\n")

#  Drop rows with null IDs
df_clean = df_raw.dropna(subset=["transaction_id", "customer_id"])

# Remove Duplicates
df_clean = df_clean.dropDuplicates(["transaction_id"])

# Fix the Target Variable (Churn)
# Convert Y, TRUE, 1 -> Yes. Convert N, FALSE, 0 -> No.
df_clean = df_clean.withColumn("churn",
    when(lower(trim(col("churn"))).isin("y", "true", "1", "yes"), "Yes")
    .when(lower(trim(col("churn"))).isin("n", "false", "0", "no"), "No")
    .otherwise(None)
)
df_clean = df_clean.dropna(subset=["churn"]) # Drop any weird values that aren't Yes/No

# Fix Invalid Dates
# Convert valid dates to timestamp. Drop rows with totally broken dates.
df_clean = df_clean.withColumn("transaction_date", to_timestamp(col("transaction_date"), "yyyy-MM-dd HH:mm:ss"))
df_clean = df_clean.dropna(subset=["transaction_date"])

# Handle Impossible Values (Outliers)
# Filter out negative amounts, negative ages, ages > 120, and visits > 100
df_clean = df_clean.filter(
    (col("total_amount") > 0) &
    (col("customer_age") > 0) &
    (col("customer_age") <= 120) &
    (col("num_visits") <= 100)
)

#  Fill Categorical Nulls and Blanks
# We replace nulls and empty strings with "Unknown"
df_clean = df_clean.fillna({
    "customer_gender": "Unknown",
    "payment_method": "Unknown",
    "store_location": "Unknown"
})
df_clean = df_clean.replace("", "Unknown", subset=["customer_gender", "store_location", "payment_method"])

# Fill Numeric Nulls (total_amount)
# We calculate the median amount and fill missing amounts with it to keep revenue accurate.
median_amount = df_clean.approxQuantile("total_amount", [0.5], 0.01)[0]
df_clean = df_clean.fillna({"total_amount": median_amount})

print(f" Cleaning Complete!")
print(f" Rows Removed: {total_rows - df_clean.count():,}")
print(f" Clean Rows Remaining: {df_clean.count():,}")

Starting Data Cleaning...

 Cleaning Complete!
 Rows Removed: 79,136
 Clean Rows Remaining: 920,864


**The** **Critical** **Handoff**

> We save the clean data as a Parquet file. Parquet is a compressed, columnar format. If we give the Data Scientist a CSV, their ML model will take 10x longer to train. Parquet is the industry standard for Big Data.



In [ ]:

df_clean.write.mode("overwrite").parquet("cleaned_retailmart.parquet")
print(" Cleaned data saved as Parquet!")

# Verify the save worked
df_test = spark.read.parquet("cleaned_retailmart.parquet")
print(f"Verification - Rows in clean file: {df_test.count():,}")

 Cleaned data saved as Parquet!
Verification - Rows in clean file: 920,864


**Auto**-**Save** **to** **Google** **Drive**

In [ ]:

from google.colab import drive
import os

#  Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define the folder path in your Drive
# You can change 'RetailMart_Project' to whatever folder name you want to create
drive_folder_path = '/content/drive/MyDrive/RetailMart_Project'

#  Create the folder if it doesn't exist yet
if not os.path.exists(drive_folder_path):
    os.makedirs(drive_folder_path)
    print(f"Created new folder: {drive_folder_path}")
else:
    print(f"Folder already exists: {drive_folder_path}")

# Define the exact path for the Parquet file
parquet_output_path = os.path.join(drive_folder_path, 'cleaned_retailmart.parquet')

#  Write the PySpark DataFrame to Google Drive
print("Saving cleaned data to Google Drive (this may take 1-2 minutes)...")
df_clean.write.mode("overwrite").parquet(parquet_output_path)

print(f"SUCCESS! Data automatically saved to Google Drive at:\n{parquet_output_path}")

Mounting Google Drive...
Mounted at /content/drive
Created new folder: /content/drive/MyDrive/RetailMart_Project
Saving cleaned data to Google Drive (this may take 1-2 minutes)...
SUCCESS! Data automatically saved to Google Drive at:
/content/drive/MyDrive/RetailMart_Project/cleaned_retailmart.parquet
